In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl datasets accelerate bitsandbytes
!pip install shutil

# 0. SCELTA DEGLI IPERPARAMETRI

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import glob
import torch
import json
import math
import shutil
import gc
import pandas as pd
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from trl import SFTTrainer, SFTConfig

# ==========================================
# 1. CONFIGURAZIONE GRID SEARCH
# ==========================================
MODEL_CHECKPOINT = "unsloth/Qwen2.5-3B-bnb-4bit"  
MAX_LEN = 2048
RELAZIONI_VALIDE = {"USA_INGREDIENTE", "USA_TECNICA", "TIPO_DI_PIATTO"}

# 5 combinazioni di iperparametri da testare
GRID = [
    {"r": 8,  "alpha": 16, "lr": 2e-4, "dropout": 0.0,  "nome": "config_1_baseline"},
    {"r": 16, "alpha": 32, "lr": 2e-4, "dropout": 0.05, "nome": "config_2_r16_dropout"},
    {"r": 32, "alpha": 64, "lr": 2e-4, "dropout": 0.05, "nome": "config_3_r32"},
    {"r": 16, "alpha": 32, "lr": 1e-4, "dropout": 0.1,  "nome": "config_4_lr_basso"},
    {"r": 16, "alpha": 16, "lr": 3e-4, "dropout": 0.0,  "nome": "config_5_lr_alto"},
]

# ==========================================
# 2. CARICAMENTO DATASET PRE-ELABORATI
# ==========================================
print("📥 Ricerca dei file generati in locale...")
tutti_i_file = glob.glob('/kaggle/input/**/*.jsonl', recursive=True)

# Trova in automatico i file basandosi sui nomi assegnati dallo script locale
FILE_TRAIN_MISTO = [f for f in tutti_i_file if "local_train_misto" in f.lower()][0]

print(f"File Train Misto: {FILE_TRAIN_MISTO}")

# Caricamento robusto (senza test finale)
dataset_train_completo = load_dataset("json", data_files=FILE_TRAIN_MISTO, split="train")

# ==========================================
# 3. SPLIT 85% TRAIN / 15% VALIDATION
# ==========================================
print("✂️ Splitting del Train set locale in Train (85%) e Validation (15%)...")
split_interno = dataset_train_completo.train_test_split(test_size=0.15, seed=42)
dataset_train = split_interno["train"]
dataset_val = split_interno["test"]

print(f"✅ Dataset caricati e splittati con successo:")
print(f"   • Train Set (misto): {len(dataset_train)} esempi")
print(f"   • Validation Set: {len(dataset_val)} esempi")

# ==========================================
# 4. TOKENIZER E PROMPT TEMPLATE
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

if getattr(tokenizer, "chat_template", None) is None:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{'<|im_start|>' + message['role'] + '\\n' + message['content'] + '<|im_end|>' + '\\n'}}"
        "{% endfor %}"
        "{% if add_generation_prompt %}{{ '<|im_start|>assistant\\n' }}{% endif %}"
    )

def applica_prompt_template(esempio):
    messaggi = [
        {"role": "system",    "content": esempio["instruction"]},
        {"role": "user",      "content": ejemplo["input"]},
        {"role": "assistant", "content": esempio["output"]},
    ]
    esempio["text"] = tokenizer.apply_chat_template(
        messaggi, tokenize=False, add_generation_prompt=False
    )
    return esempio

dataset_train = dataset_train.map(applica_prompt_template)
dataset_val   = dataset_val.map(applica_prompt_template)

# ==========================================
# 5. FUNZIONI DI VALUTAZIONE NER / RE
# ==========================================
def parse_triple(testo):
    triple = set()
    for riga in testo.strip().split("\n"):
        parti = [p.strip() for p in riga.split("|")]
        if len(parti) == 3 and parti[1] in RELAZIONI_VALIDE:
            triple.add(tuple(parti))
    return triple

def estrai_entita(triple):
    """Estrae soggetti e oggetti come entità NER"""
    return {t[0] for t in triple} | {t[2] for t in triple}

def calcola_metriche(predette_str, attese_str):
    pred_triple  = parse_triple(predette_str)
    vere_triple  = parse_triple(attese_str)
    pred_entita  = estrai_entita(pred_triple)
    vere_entita  = estrai_entita(vere_triple)

    def prf(pred_set, vero_set):
        if not pred_set and not vero_set:
            return 1.0, 1.0, 1.0
        if not pred_set or not vero_set:
            return 0.0, 0.0, 0.0
        vp = len(pred_set & vero_set)
        p  = vp / len(pred_set)
        r  = vp / len(vero_set)
        f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
        return p, r, f1

    re_p,  re_r,  re_f1  = prf(pred_triple,  vere_triple)   # Relation Extraction
    ner_p, ner_r, ner_f1 = prf(pred_entita,  vere_entita)   # NER

    acc_parziale = (
        len(pred_triple & vere_triple) / len(vere_triple)
        if vere_triple else 0.0
    )

    return {
        "re_precision":  re_p,
        "re_recall":     re_r,
        "re_f1":         re_f1,
        "ner_precision": ner_p,
        "ner_recall":    ner_r,
        "ner_f1":        ner_f1,
        "acc_parziale":  acc_parziale,
    }

def valuta_modello(model, tokenizer, dataset_val, n_campioni=80):
    """Genera predizioni su un campione del validation set e calcola le metriche"""
    FastLanguageModel.for_inference(model)
    model.eval()

    campione = dataset_val.select(range(min(n_campioni, len(dataset_val))))
    risultati = []

    with torch.no_grad():
        for esempio in campione:
            messaggi = [
                {"role": "system", "content": esempio["instruction"]},
                {"role": "user",   "content": esempio["input"]},
            ]
            prompt = tokenizer.apply_chat_template(
                messaggi, tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer([prompt], return_tensors="pt", truncation=True,
                               max_length=MAX_LEN).to("cuda")

            output = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )
            risposta = tokenizer.batch_decode(output, skip_special_tokens=True)[0]

            tag = "assistant\n"
            predetta = risposta.split(tag)[-1].strip() if tag in risposta else risposta
            risultati.append(calcola_metriche(predetta, esempio["output"]))

    # Calcola perplexity sul testo completo (train mode)
    model.train()
    losses = []
    with torch.no_grad():
        for esempio in campione:
            ids = tokenizer(
                esempio["text"], return_tensors="pt",
                truncation=True, max_length=MAX_LEN
            ).to("cuda")
            out  = model(**ids, labels=ids["input_ids"])
            losses.append(out.loss.item())
    perplexity = math.exp(sum(losses) / len(losses))

    medie = {k: sum(r[k] for r in risultati) / len(risultati) for k in risultati[0]}
    medie["perplexity"] = perplexity
    return medie

# ==========================================
# 6. ESECUZIONE GRID SEARCH
# ==========================================
tutti_i_risultati = []
usa_bf16 = torch.cuda.is_bf16_supported()

for cfg in GRID:
    print(f"\n{'='*55}")
    print(f"🔬 Training Grid: {cfg['nome']}")
    print(f"   r={cfg['r']} | alpha={cfg['alpha']} | lr={cfg['lr']} | dropout={cfg['dropout']}")
    print(f"{'='*55}")

    # Carica modello base pulito per ogni configurazione della griglia
    model, _ = FastLanguageModel.from_pretrained(
        model_name=MODEL_CHECKPOINT,
        max_seq_length=MAX_LEN,
        dtype=None,
        load_in_4bit=True,
    )
    model_lora = FastLanguageModel.get_peft_model(
        model,
        r=cfg["r"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=cfg["alpha"],
        lora_dropout=cfg["dropout"],
        bias="none",
        random_state=3407,
    )

    cartella_checkpoint_tmp = f"./tmp_grid_{cfg['nome']}"

    sft_config = SFTConfig(
        output_dir=cartella_checkpoint_tmp,
        learning_rate=cfg["lr"],
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        num_train_epochs=3,           
        fp16=not usa_bf16,
        bf16=usa_bf16,
        optim="adamw_8bit",
        dataset_text_field="text",
        report_to="none",
        logging_strategy="epoch",
        eval_strategy="epoch",
        save_strategy="no", 
        disable_tqdm=False,
    )

    trainer = SFTTrainer(
        model=model_lora,
        train_dataset=dataset_train,
        eval_dataset=dataset_val,
        processing_class=tokenizer,
        args=sft_config,
    )
    
    trainer.train()

    # Valutazione Approfondita a fine dei 3 Epochs
    print(f"📊 Calcolo metriche per {cfg['nome']}...")
    metriche = valuta_modello(model_lora, tokenizer, dataset_val, n_campioni=80)
    metriche["config"] = cfg["nome"]
    metriche.update(cfg)
    tutti_i_risultati.append(metriche)

    print(f"   RE  → P:{metriche['re_precision']:.3f} R:{metriche['re_recall']:.3f} F1:{metriche['re_f1']:.3f}")
    print(f"   NER → P:{metriche['ner_precision']:.3f} R:{metriche['ner_recall']:.3f} F1:{metriche['ner_f1']:.3f}")
    print(f"   Acc parziale: {metriche['acc_parziale']:.3f} | Perplexity: {metriche['perplexity']:.3f}")

    # Pulizia della memoria VRAM ed eliminazione delle cartelle temporanee
    del model, model_lora, trainer
    gc.collect()
    torch.cuda.empty_cache()
    if os.path.exists(cartella_checkpoint_tmp):
        shutil.rmtree(cartella_checkpoint_tmp)

# ==========================================
# 7. RIEPILOGO E SALVATAGGIO RISULTATI
# ==========================================
print(f"\n{'='*55}")
print("📋 RIEPILOGO GRID SEARCH COMPLETATO")
print(f"{'='*55}")
print(f"{'Config':<28} {'RE-F1':>6} {'NER-F1':>7} {'Acc':>6} {'PPL':>7}")
print("-"*55)
for r in sorted(tutti_i_risultati, key=lambda x: x["re_f1"], reverse=True):
    print(f"{r['config']:<28} {r['re_f1']:>6.3f} {r['ner_f1']:>7.3f} "
          f"{r['acc_parziale']:>6.3f} {r['perplexity']:>7.2f}")

# Salva i risultati aggregati in formato JSON
with open("grid_search_risultati.json", "w", encoding="utf-8") as f:
    json.dump(tutti_i_risultati, f, ensure_ascii=False, indent=2)
print("\n💾 Risultati salvati in grid_search_risultati.json")

Sebbene lo script sia strutturato per eseguire la Grid Search in modo sequenziale, per motivi di tempistica e ottimizzazione delle risorse abbiamo scelto di eseguire i test in parallelo. Ogni membro del team ha avviato una singola configurazione della griglia sul proprio Kaggle.

Di conseguenza, l'output cumulativo di questa cella non è disponibile in questo notebook. I risultati completi di tutte le configurazioni e la giustificazione della scelta del modello finale sono stati aggregati e sono consultabili nella relazione a pagina 15 (Tabella 2).

#### Una volta scelta la configurazione, abbiamo avviamo le celle presenti sotto.

# 1. SET UP AMBIENTE, CARICAMENTO E PREPARAZIONE DEI DATI

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import glob
import torch
import json
import math
import shutil
import gc
import pandas as pd
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from trl import SFTTrainer, SFTConfig

# ==========================================
# 1. CONFIGURAZIONE PARAMETRI
# ==========================================
MODEL_CHECKPOINT = "unsloth/Qwen2.5-3B-bnb-4bit"
NOME_CARTELLA_SALVATAGGIO = "modello_triple_culinarie_finale"
MAX_LEN = 2048
RELAZIONI_VALIDE = {"USA_INGREDIENTE", "USA_TECNICA", "TIPO_DI_PIATTO"}

R = 8
ALPHA = 16
LR = 2e-4
DROPOUT = 0.0
NOME_CONFIG = "config_1_dataset_misto"

# ==========================================
# 2. CARICAMENTO DATASETS
# ==========================================
print("📥 Ricerca dei file generati in locale...")
tutti_i_file = glob.glob('/kaggle/input/**/*.jsonl', recursive=True)

FILE_TRAIN_MISTO = [f for f in tutti_i_file if "local_train_misto" in f.lower()][0]
FILE_TEST_FINALE = [f for f in tutti_i_file if "local_test_finale" in f.lower()][0]

print(f"File Train Misto: {FILE_TRAIN_MISTO}")
print(f"File Test Finale: {FILE_TEST_FINALE}")

dataset_train_completo = load_dataset("json", data_files=FILE_TRAIN_MISTO, split="train")
dataset_test_ood = load_dataset("json", data_files=FILE_TEST_FINALE, split="train")

# ==========================================
# 3. SPLIT 85% TRAIN / 15% VALIDATION
# ==========================================
print("✂️ Splitting del Train set locale in Train (85%) e Validation (15%)...")
split_interno = dataset_train_completo.train_test_split(test_size=0.15, seed=42)
dataset_train = split_interno["train"]
dataset_val = split_interno["test"]

print(f"✅ Dataset caricati e splittati con successo:")
print(f"   • Train Set (misto): {len(dataset_train)} esempi")
print(f"   • Validation Set: {len(dataset_val)} esempi")
print(f"   • Test Set (OOD puro): {len(dataset_test_ood)} esempi")

# ==========================================
# 4. TOKENIZER E PROMPT TEMPLATE
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

if getattr(tokenizer, "chat_template", None) is None:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{'<|im_start|>' + message['role'] + '\\n' + message['content'] + '<|im_end|>' + '\\n'}}"
        "{% endfor %}"
        "{% if add_generation_prompt %}{{ '<|im_start|>assistant\\n' }}{% endif %}"
    )

def applica_prompt_template(esempio):
    messaggi = [
        {"role": "system",    "content": esempio["instruction"]},
        {"role": "user",      "content": esempio["input"]},
        {"role": "assistant", "content": esempio["output"]},
    ]
    esempio["text"] = tokenizer.apply_chat_template(
        messaggi, tokenize=False, add_generation_prompt=False
    )
    return esempio

dataset_train = dataset_train.map(applica_prompt_template)
dataset_val   = dataset_val.map(applica_prompt_template)
dataset_test_ood = dataset_test_ood.map(applica_prompt_template)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
📥 Ricerca dei file generati in locale...
File Train Misto: /kaggle/input/datasets/paolaapap/local-train-misto/local_train_misto.jsonl
File Test Finale: /kaggle/input/datasets/paolaapap/local-test-finale/local_test_finale.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

✂️ Splitting del Train set locale in Train (85%) e Validation (15%)...
✅ Dataset caricati e splittati con successo:
   • Train Set (misto): 977 esempi
   • Validation Set: 173 esempi
   • Test Set (OOD puro): 50 esempi


config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

Map:   0%|          | 0/977 [00:00<?, ? examples/s]

Map:   0%|          | 0/173 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

# 2. TRAINING

In [3]:
print(f"\n{'='*55}")
print(f"🔬 Inizio Training Misto | r={R} | alpha={ALPHA}")
print(f"{'='*55}")

# Inizializzazione Modello Unsloth
model, _ = FastLanguageModel.from_pretrained(
    model_name=MODEL_CHECKPOINT,
    max_seq_length=MAX_LEN,
    dtype=None,
    load_in_4bit=True,
)

model_lora = FastLanguageModel.get_peft_model(
    model,
    r=R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=ALPHA,
    lora_dropout=DROPOUT,
    bias="none",
    random_state=3407,
)

usa_bf16 = torch.cuda.is_bf16_supported()
cartella_checkpoint = f"./tmp_{NOME_CONFIG}"

sft_config = SFTConfig(
    output_dir=cartella_checkpoint,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    num_train_epochs=3,           
    fp16=not usa_bf16,
    bf16=usa_bf16,
    optim="adamw_8bit",
    dataset_text_field="text",
    report_to="none",
    
    # Configurazione di log per vedere avanzamento e tempi ad ogni epoca
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=1,
    disable_tqdm=False, # Forza la barra di avanzamento con i tempi stimati
    save_total_limit=2,
)

trainer = SFTTrainer(
    model=model_lora,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    processing_class=tokenizer,
    args=sft_config,
)

trainer.train()


🔬 Inizio Training Misto | r=8 | alpha=16
==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 977 | Num Epochs = 3 | Total steps = 735
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 14,966,784 of 3,100,905,472 (0.48% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.776314,0.624879
2,0.567303,0.588512
3,0.498253,0.589304


TrainOutput(global_step=735, training_loss=0.613956643448395, metrics={'train_runtime': 7905.3242, 'train_samples_per_second': 0.371, 'train_steps_per_second': 0.093, 'total_flos': 5.598120460470682e+16, 'train_loss': 0.613956643448395, 'epoch': 3.0})

# 3. SALVATAGGIO LoRA ED ESPORTAZIONE ZIP 

In [ ]:
# Questo salverà solo i file lora_weights (adapter_config.json e adapter_model.safetensors)
model_lora.save_pretrained(NOME_CARTELLA_SALVATAGGIO)
tokenizer.save_pretrained(NOME_CARTELLA_SALVATAGGIO)

CARTELLA_EXPORT = "/kaggle/working/export_loRA"
os.makedirs(CARTELLA_EXPORT, exist_ok=True)

# Copiamo la cartella finale contenente esclusivamente i pesi di LoRA
shutil.copytree(NOME_CARTELLA_SALVATAGGIO, os.path.join(CARTELLA_EXPORT, NOME_CARTELLA_SALVATAGGIO), dirs_exist_ok=True)

# Genera l'archivio ZIP 
shutil.make_archive("/kaggle/working/lora_weights_culinarie", "zip", root_dir=CARTELLA_EXPORT)
shutil.rmtree(CARTELLA_EXPORT)

# Liberiamo la VRAM per la cella successiva di test
del trainer
torch.cuda.empty_cache()
gc.collect()

In [8]:
from IPython.display import FileLink

FileLink(r'lora_weights_culinarie.zip')

/kaggle/working/lora_weights_culinarie.zip

# 4. GENERAZIONE E VALUTAZIONE SUL TEST SET OOD 

In [9]:
# ==========================================
# DEFINIZIONE FUNZIONI LOGICHE DI VALUTAZIONE 
# ==========================================
def parse_triple(testo):
    """Estrae le triple uniche ripulendo gli spazi e controllando la relazione"""
    triple = set()
    for riga in testo.strip().split("\n"):
        parti = [p.strip() for p in riga.split("|")]
        if len(parti) == 3 and parti[1] in RELAZIONI_VALIDE:
            triple.add(tuple(parti))
    return triple

def estrai_entita(triple):
    """
    Estrae le entità accoppiando la stringa testuale alla sua classe semantica.
    Usa la relazione della tripla per determinare la classe dell'Oggetto.
    """
    entita = set()
    for sogg, rel, ogg in triple:
        # Il soggetto nelle nostre ricette è sempre il nome della Ricetta
        entita.add((sogg, "Ricetta")) 
        
        # L'oggetto cambia classe semantica in base alla relazione formale
        if rel == "USA_INGREDIENTE":
            entita.add((ogg, "Ingrediente"))
        elif rel == "USA_TECNICA":
            entita.add((ogg, "Tecnica"))
        elif rel == "TIPO_DI_PIATTO":
            entita.add((ogg, "Tipo_Piatto"))
        else:
            entita.add((ogg, "Altro"))
    return entita

def calcola_precision_recall_f1(pred_set, vero_set):
    """Calcola formalmente Precision, Recall e F1-Score per un set di elementi"""
    if not pred_set and not vero_set:
        return 1.0, 1.0, 1.0
    if not pred_set or not vero_set:
        return 0.0, 0.0, 0.0
    
    veri_positivi = len(pred_set & vero_set)
    precision = veri_positivi / len(pred_set)
    recall = veri_positivi / len(vero_set)
    
    f1_score = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1_score

def calcola_metriche_globali(predette_str, attese_str):
    """Calcola metriche multilivello: Strict Triplet (RE) e NER"""
    pred_triple = parse_triple(predette_str)
    vere_triple = parse_triple(attese_str)
    
    pred_entita = estrai_entita(pred_triple)
    vere_entita = estrai_entita(vere_triple)

    # 1. LIVELLO: Strict Triplet (Soggetto, Relazione, Oggetto devono coincidere esattamente)
    st_p, st_r, st_f1 = calcola_precision_recall_f1(pred_triple, vere_triple)
    
    # 2. LIVELLO: NER (Confini testuali ed etichetta semantica della classe)
    ner_p, ner_r, ner_f1 = calcola_precision_recall_f1(pred_entita, vere_entita)

    return {
        "strict_triplet_precision": st_p,
        "strict_triplet_recall": st_r,
        "strict_triplet_f1": st_f1,
        "ner_precision": ner_p,
        "ner_recall": ner_r,
        "ner_f1": ner_f1
    }

def valuta_modello(model, tokenizer, dataset_da_valutare, n_campioni=50):
    FastLanguageModel.for_inference(model)
    model.eval()

    campione = dataset_da_valutare.select(range(min(n_campioni, len(dataset_da_valutare))))
    risultati = []

    with torch.no_grad():
        for indice, esempio in enumerate(campione):
            if (indice + 1) % 10 == 0 or (indice + 1) == len(campione):
                print(f"🔄 Generazione in corso: {indice + 1}/{len(campione)} campioni processati...")
                
            messaggi = [
                {"role": "system", "content": esempio["instruction"]},
                {"role": "user",   "content": esempio["input"]},
            ]
            prompt = tokenizer.apply_chat_template(messaggi, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer([prompt], return_tensors="pt", truncation=True, max_length=MAX_LEN).to("cuda")

            output = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
            risposta = tokenizer.batch_decode(output, skip_special_tokens=True)[0]

            tag = "assistant\n"
            predetta = risposta.split(tag)[-1].strip() if tag in risposta else risposta
            risultati.append(calcola_metriche_globali(predetta, esempio["output"]))

    # Calcolo Perplexity
    losses = []
    with torch.no_grad():
        for esempio in campione:
            ids = tokenizer(esempio["text"], return_tensors="pt", truncation=True, max_length=MAX_LEN).to("cuda")
            out  = model(**ids, labels=ids["input_ids"])
            losses.append(out.loss.item())
    perplexity = math.exp(sum(losses) / len(losses))

    # Media delle metriche su tutti i campioni
    medie = {k: sum(r[k] for r in risultati) / len(risultati) for k in risultati[0]}
    medie["perplexity"] = perplexity
    return medie

# ==========================================
# ESECUZIONE VALUTAZIONE SUL TEST SET OOD 
# ==========================================
print("\n🎯 Avvio del testing finale sul Test Set OOD (50 elementi)...")
metriche_ood = valuta_modello(model_lora, tokenizer, dataset_test_ood, n_campioni=50)

print(f"\n{'='*70}")
print(f"🌟 RISULTATO VALUTAZIONE FORMALE SUL TEST SET (OOD)")
print(f"{'='*70}")
print(f" 🟩 NER LEVEL:")
print(f"   • NER Precision: {metriche_ood['ner_precision']:.3f}")
print(f"   • NER Recall:    {metriche_ood['ner_recall']:.3f}")
print(f"   • NER F1-Score:  {metriche_ood['ner_f1']:.3f}  <--")
print(f" 🟦 RE LEVEL (STRICT TRIPLET):")
print(f"   • Strict Triplet Precision: {metriche_ood['strict_triplet_precision']:.3f}")
print(f"   • Strict Triplet Recall:    {metriche_ood['strict_triplet_recall']:.3f}")
print(f"   • Strict Triplet F1-Score:  {metriche_ood['strict_triplet_f1']:.3f}  <-- (Metrica Principale)")
print(f" ⚙️ INTRINSIC METRICS:")
print(f"   • Perplexity del Modello:   {metriche_ood['perplexity']:.3f}")
print(f"{'='*70}\n")

# Salva report JSON finale
with open("metriche_test_finale_ood.json", "w", encoding="utf-8") as f:
    json.dump(metriche_ood, f, ensure_ascii=False, indent=2)
print("📝 File 'metriche_test_finale_ood.json' pronto per i tuoi report.")

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🎯 Avvio del testing finale sul Test Set OOD (50 elementi)...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5

🔄 Generazione in corso: 10/50 campioni processati...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

🔄 Generazione in corso: 40/50 campioni processati...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

🔄 Generazione in corso: 50/50 campioni processati...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


🌟 RISULTATO VALUTAZIONE FORMALE SUL TEST SET (OOD)
 🟩 NER LEVEL:
   • NER Precision: 0.559
   • NER Recall:    0.972
   • NER F1-Score:  0.702  <--
 🟦 RE LEVEL (STRICT TRIPLET):
   • Strict Triplet Precision: 0.538
   • Strict Triplet Recall:    0.970
   • Strict Triplet F1-Score:  0.683  <-- (Metrica Principale)
 ⚙️ INTRINSIC METRICS:
   • Perplexity del Modello:   1.217

📝 File 'metriche_test_finale_ood.json' pronto per i tuoi report.


# 5. MERGE DEL MODELLO BASE + PESI LoRA

In [3]:
import torch
from unsloth import FastLanguageModel

PERCORSO_LORA_INPUT = "/kaggle/input/datasets/paolaapap/miei-pesi-lora/modello_triple_culinarie_finale"
NOME_CARTELLA_MERGED = "modello_finale_merged_ricette"
MAX_LEN = 2048

print("🔄 Caricamento del modello base + pesi LoRA...")


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = PERCORSO_LORA_INPUT,
    max_seq_length = MAX_LEN,
    dtype = None,                    
    load_in_4bit = False,             
)


print("💾 Esecuzione del Merge e salvataggio a 16-bit...")
model.save_pretrained_merged(
    NOME_CARTELLA_MERGED, 
    tokenizer, 
    save_method = "merged_16bit" # Unisce i pesi e salva il modello completo
)

print("✅ Fatto! Modello unificato pronto in:", NOME_CARTELLA_MERGED)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🔄 Caricamento del modello base + pesi LoRA...
==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

...

Checking cache directory for required files...



Unsloth: Copying 2 files from cache to `modello_finale_merged_ricette`:   0%|          | 0/2 [00:00<?, ?it/s]
Unsloth: Copying 2 files from cache to `modello_finale_merged_ricette`:  50%|█████     | 1/2 [00:04<00:04,  4.25s/it]
Unsloth: Copying 2 files from cache to `modello_finale_merged_ricette`: 100%|██████████| 2/2 [00:06<00:00,  3.07s/it]


Successfully copied all 2 files from cache to `modello_finale_merged_ricette`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 12768.05it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:41<00:00, 20.95s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/modello_finale_merged_ricette`
✅ Fatto! Modello unificato pronto in: modello_finale_merged_ricette


In [4]:
import shutil
from IPython.display import FileLink

shutil.make_archive("/kaggle/working/modello_completo_ricette", "zip", root_dir="modello_finale_merged_ricette")
FileLink(r'modello_completo_ricette.zip')

/kaggle/working/modello_completo_ricette.zip

# 6. INFERENZA E TEST SU UN POST REALE (generato dal nostro sistema agentico) 

In [11]:
import re
import torch
from unsloth import FastLanguageModel

# =====================================================================
# 1. INIZIALIZZAZIONE E CARICAMENTO DEL MODELLO FINE TUNATO
# =====================================================================
PERCORSO_NUOVO_DATASET = "/kaggle/working/modello_finale_merged_ricette"
MAX_LEN = 2048

print("🔄 Caricamento in corso del modello fine-tunato unificato...")
# Rimosso 'fix_mistral_regex' da qui per evitare il TypeError
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = PERCORSO_NUOVO_DATASET,
    max_seq_length = MAX_LEN,
    dtype = None,                     
    load_in_4bit = True,              
)

# =====================================================================
# 1b. RIPRISTINO DEL CHAT TEMPLATE (RISOLVE IL VALUEERROR)
# =====================================================================
if tokenizer.chat_template is None:
    print("⚠️ Chat template non trovato nel modello locale. Configuro il template ChatML di default...")
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n' }}"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "{{ '<|im_start|>assistant\n' }}"
        "{% endif %}"
    )

FastLanguageModel.for_inference(model) 
model.eval()

# =====================================================================
# 2. PREPARAZIONE DEL INPUT E PROMPT
# =====================================================================
testo_ricetta = """La cucina siciliana è famosa per le sue ricette deliziose e autentiche, e uno dei piatti più rappresentativi di questa tradizione è senza dubbio la "Norma". Questo piatto, che prende il nome dall'opera lirica di Vincenzo Bellini, è un vero e proprio tributo alla cucina siciliana e ai suoi ingredienti tipici. In questo articolo, vi guiderò nella preparazione dei Tortiglioni alla Norma, un piatto che combina la pasta con le melanzane, il pomodoro e la ricotta salata.

Per iniziare, è importante avere a disposizione gli ingredienti necessari. Ecco la lista degli ingredienti obbligatori per la preparazione dei Tortiglioni alla Norma:

* Melanzane (1 grande) (Circa 0.65 € (per 0.5 Kg))
* Pomodori pelati (400 g) (Circa 1.08 € (per 0.5 Kg))
* Basilico (q.b.) (Circa 0.90 € (per 0.05 Kg))
* Olio extravergine d'oliva (q.b.) (Circa 0.57 € (per 0.05 Kg))
* Aglio (2 spicchi) (Circa 0.20 € (per 0.05 Kg))
* Sale (q.b.) (Circa 0.004 € (per 0.005 Kg))
* Pepe nero (q.b.) (Circa 0.175 € (per 0.005 Kg))
* Ricotta salata (200 g) (Circa 0.43 € (per 0.05 Kg))
* Tortiglioni (400 g) (Circa 0.19 € (per 0.05 Kg))
* Parmigiano Reggiano DOP (100 g) (Circa 1.10 € (per 0.05 Kg))

La preparazione dei Tortiglioni alla Norma inizia con la pulizia e il taglio delle melanzane. È importante tagliare le melanzane a fette sottili e lasciarle riposare per circa 30 minuti in modo che perdano il loro eccesso di acqua [Fonte: https://cookingitalians.com/authentic-pasta-alla-norma-a-taste-of-sicily/]. Successivamente, le melanzane vanno fritte in abbondante olio caldo fino a quando non sono dorate [Fonte: Knowledge Graph].

Mentre le melanzane sono in cottura, possiamo preparare il sugo. In un grande tegame, soffriggiamo l'aglio e l'olio, poi aggiungiamo la passata di pomodoro e cuociamo per circa 20 minuti [Fonte: https://cookingitalians.com/authentic-pasta-alla-norma-a-taste-of-sicily/]. Infine, aggiungiamo il basilico fresco e il sale [Fonte: Knowledge Graph].

La pasta, ovviamente, è un elemento fundamental di questo piatto. I Tortiglioni vanno cotti al dente in abbondante acqua salata, poi scolati e uniti al sugo [Fonte: Knowledge Graph]. A questo punto, aggiungiamo le melanzane fritte e la ricotta salata grattugiata sopra [Fonte: https://cookingitalians.com/authentic-pasta-alla-norma-a-taste-of-sicily/].

È importante non cuocere troppo la pasta e non utilizzare mozzarella o altri formaggi filanti, in quanto ciò altererebbe il sapore tradicional della ricetta [Fonte: https://cookingitalians.com/authentic-pasta-alla-norma-a-taste-of-sicily/]. La Norma è un piatto che richiede semplicità e autenticità, quindi è fondamentale respecter le tradizioni culinarie siciliane.

In conclusione, i Tortiglioni alla Norma sono un piatto delizioso e autentico che rappresenta al meglio la cucina siciliana. Con gli ingredienti giusti e la preparazione appropriata, potrete gustare un piatto che è stato apprezzato per generazioni.

💡 I Consigli dello Chef
Se desiderate una variante più sostenibile e naturale, potete sostituire gli ingredienti con le loro alternative BIO. Ad esempio, potete utilizzare melanzane BIO, pomodori BIO, basilico BIO, olio extravergine di oliva BIO, aglio BIO, sale fino BIO, pepe nero BIO, ricotta BIO, pasta di semola BIO e parmigiano reggiano BIO. Inoltre, se siete intolleranti al lattosio, potete sostituire la ricotta salata con una variante senza lattosio.

🍷 L'Abbinamento Perfetto
Chianti Classico, un vino rosso toscano con note di frutta rossa e spezie, che si abbina bene con i sapori decisi della ricetta."""

istruzione_fissa = "Analizza il testo della ricetta ed estrai le relazioni nel formato rigido: Soggetto | RELAZIONE | Oggetto. Usa solo i termini approvati: USA_INGREDIENTE, USA_TECNICA, TIPO_DI_PIATTO."

messaggi_inferenza = [
    {"role": "system", "content": istruzione_fissa},
    {"role": "user",   "content": testo_ricetta},
]

prompt_tokenizzato = tokenizer.apply_chat_template(
    messaggi_inferenza, 
    tokenize=False, 
    add_generation_prompt=True
)

inputs = tokenizer([prompt_tokenizzato], return_tensors="pt", truncation=True, max_length=MAX_LEN).to("cuda")

# Gestione dei token di stop
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>"),
    tokenizer.convert_tokens_to_ids("<|end_of_text|>"), 
    tokenizer.convert_tokens_to_ids("<|im_end|>") 
]
terminatori_validi = [t for t in terminators if t is not None]


# =====================================================================
# 3. GENERAZIONE ED ESTRAZIONE DELLE TRIPLE
# =====================================================================
print("🧠 Generazione delle triple in corso...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=350,
        use_cache=True,
        eos_token_id=terminatori_validi,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.0, 
        do_sample=False,       
    )

risposta_pulita = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

righe_valide = []
for riga in risposta_pulita.split('\n'):
    riga = riga.strip()
    if riga.count('|') == 2:
        parti = riga.split('|')
        soggetto = parti[0].strip()
        relazione = parti[1].strip()
        oggetto = parti[2].strip()
      
        # Pulizia caratteri speciali e artefatti di testo
        oggetto = re.sub(r'[^\x00-\x7F\xC0-\xFF]', '', oggetto)
        oggetto = re.sub(r'(?i)\bassistant\b', '', oggetto)
        oggetto = re.sub(r'\b\w*([a-zA-Z])\1{2,}\w*\b', '', oggetto)
        oggetto = oggetto.strip()
        
        if oggetto:
            tripla_pulita = f"{soggetto} | {relazione} | {oggetto}"
            righe_valide.append(tripla_pulita)
    else:
        continue 

# =====================================================================
# 4. STAMPA DEI RISULTATI
# =====================================================================
print(f"\n{'='*65}")
print("🚀 OUTPUT GENERATO DAL MODELLO FINE-TUNATO:")
print(f"{'='*65}")
if righe_valide:
    print('\n'.join(righe_valide))
else:
    print("⚠️ Nessuna tripla valida estratta nel formato specificato.")
    print("Testo grezzo generato per debug:\n", risposta_pulita) 
print(f"{'='*65}\n")

🔄 Caricamento in corso del modello fine-tunato unificato...
==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

The tokenizer you are loading from '/kaggle/working/modello_finale_merged_ricette' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/kaggle/working/modello_finale_merged_ricette' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Both `max_new_tokens` (=350) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⚠️ Chat template non trovato nel modello locale. Configuro il template ChatML di default...
🧠 Generazione delle triple in corso...

🚀 OUTPUT GENERATO DAL MODELLO FINE-TUNATO:
Tortiglioni alla Norma | TIPO_DI_PIATTO | Primo piatto
Tortiglioni alla Norma | USA_INGREDIENTE | Aglio
Tortiglioni alla Norma | USA_INGREDIENTE | Basilico
Tortiglioni alla Norma | USA_INGREDIENTE | Melanzane
Tortiglioni alla Norma | USA_INGREDIENTE | Olio extravergine d'oliva
Tortiglioni alla Norma | USA_INGREDIENTE | Parmigiano Reggiano DOP
Tortiglioni alla Norma | USA_INGREDIENTE | Pepe nero
Tortiglioni alla Norma | USA_INGREDIENTE | Pomodori pelati
Tortiglioni alla Norma | USA_INGREDIENTE | Ricotta salata
Tortiglioni alla Norma | USA_INGREDIENTE | Sale
Tortiglioni alla Norma | USA_INGREDIENTE | Tortiglioni
Tortiglioni alla Norma | USA_TECNICA | Frittura
Tortiglioni alla Norma | USA_TECNICA | Cottura al dente
Tortiglioni alla Norma | USA_TECNICA | Soffriggere
Tortiglioni alla Norma | USA_TECNICA | Cottura in ac